# HW3 — Part 1: HedgeMaze Multi-Agent VDN
### Cooperative MARL · Value Decomposition Networks · ε-greedy with Decay
**Yogeshvar Reddy Kallam** · IST 597 Deep RL · Penn State Spring 2025

---

## A-maze-ing Adventure — VDN Q-Learning

Two agents (Alice & Bob) navigate a 7×7 hedge maze and must **meet** (occupy the same cell).

**VDN decomposition:**  
`Q_tot(s, a_Alice, a_Bob) = Q_Alice(s, a_Alice) + Q_Bob(s, a_Bob)`

Each agent has its own Q-table (nn.Parameter). The shared reward (+1 on meeting) backpropagates through the additive sum, implicitly coordinating both agents.

ε decays **1.0 → 0.05** over 15,000 episodes.

In [ ]:
import random, numpy as np, torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from gymnasium import spaces

DEVICE = torch.device("cpu")

class HedgeMaze:
    HEDGE = ["*******","*     *","* *** *","* *   *","*   * *","* *** *","*     *","*******"]
    AGENTS = ["Alice", "Bob"]
    def __init__(self):
        self.H=len(self.HEDGE); self.W=len(self.HEDGE[0]); self.SIZE=self.H*self.W
        self.possible_agents=self.AGENTS; self.agents=[]
    def _valid(self,y,x): return 0<=y<self.H and 0<=x<self.W and self.HEDGE[y][x]==" "
    def _rand(self,ex=None):
        while True:
            x,y=random.randrange(self.W),random.randrange(self.H)
            loc=y*self.W+x
            if self._valid(y,x) and loc!=ex: return loc
    def _obs(self): jt=self.locs[0]*self.SIZE+self.locs[1]; return {a:jt for a in self.AGENTS}
    def action_space(self,a): return spaces.Discrete(4)
    def observation_space(self,a): return spaces.Discrete(self.SIZE*self.SIZE)
    def reset(self):
        self.agents=list(self.AGENTS); self.locs=[self._rand()]; self.locs.append(self._rand(self.locs[0]))
        return self._obs(),{}
    def _move(self,loc,a):
        x,y=loc%self.W,loc//self.W
        dx,dy=[(-1,0),(1,0),(0,1),(0,-1)][a]; nx,ny=x+dx,y+dy
        return (ny*self.W+nx) if self._valid(ny,nx) else loc
    def step(self,actions):
        for i,ag in enumerate(self.AGENTS): self.locs[i]=self._move(self.locs[i],actions[ag])
        done=self.locs[0]==self.locs[1]; r=1. if done else 0.
        if done: self.agents=[]
        return self._obs(),{a:r for a in self.AGENTS},{a:done for a in self.AGENTS},{a:False for a in self.AGENTS},{}
    def render(self):
        g=[list(row) for row in self.HEDGE]
        for i,l in enumerate(self.locs): g[l//self.W][l%self.W]=["A","B"][i]
        print("\n".join("".join(r) for r in g))

class QTable(nn.Module):
    def __init__(self,ns,na): super().__init__(); self.Q=nn.Parameter(torch.zeros(ns,na))
    def forward(self,s): return self.Q[s]

env=HedgeMaze(); agents=env.possible_agents
models={a:QTable(env.observation_space(a).n, env.action_space(a).n).to(DEVICE) for a in agents}
optimizer=optim.Adam([p for m in models.values() for p in m.parameters()], lr=0.0005)
loss_fn=nn.MSELoss()

EPISODES=25_000; EPS_START=1.0; EPS_END=0.05; EPS_DECAY=15_000; GAMMA=0.9
lengths=[]

for ep in range(EPISODES):
    obs,_=env.reset()
    obs_t={a:torch.tensor(obs[a],dtype=torch.long,device=DEVICE) for a in agents}
    eps=max(EPS_END, EPS_START-(EPS_START-EPS_END)*(ep/EPS_DECAY))
    done=False; length=0

    while not done:
        length+=1; actions={}; q_cur=[]
        for ag in env.agents:
            a=env.action_space(ag).sample() if random.random()<eps else torch.argmax(models[ag](obs_t[ag])).item()
            actions[ag]=a; q_cur.append(models[ag](obs_t[ag])[a])
        next_obs,rews,terms,_,_=env.step(actions)
        r=rews[agents[0]]; done=any(terms.values())
        next_t={a:torch.tensor(next_obs[a],dtype=torch.long,device=DEVICE) for a in agents}
        with torch.no_grad():
            q_tgt=(torch.tensor(r,dtype=torch.float32) if done
                   else r+GAMMA*sum(torch.max(models[a](next_t[a])) for a in agents))
        loss=loss_fn(torch.stack(q_cur).sum(), q_tgt)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        obs_t=next_t

    lengths.append(length)
    if (ep+1)%2500==0: print(f"Ep {ep+1:>6} | ε={eps:.3f} | avg steps={np.mean(lengths[-2500:]):.1f}")

print("\n✅ VDN Training complete.")
w=500; plt.figure(figsize=(10,4))
plt.plot(np.convolve(lengths,np.ones(w)/w,'valid')); plt.xlabel("Episode")
plt.ylabel("Steps to meet"); plt.title("HedgeMaze VDN — Steps to meet"); plt.tight_layout(); plt.show()
